# 01 — Data Audit

Raw LendingClub loan data audit: shape, dtypes, missingness, and the resolved-outcome target definition.

All logic lives in `credit_risk.data` — this notebook only calls it and narrates the findings.

In [ ]:
from credit_risk.config import settings
from credit_risk.data import clean_and_label, generate_audit_table, load_raw_data

## Load raw data

In [ ]:
df = load_raw_data(settings.raw_data_path)
print("Raw shape:", df.shape)

## Column-level audit

`generate_audit_table` reports dtype, missing count/percent, and cardinality for every raw column.

In [ ]:
audit_df = generate_audit_table(df)
audit_df.sort_values("missing_percent", ascending=False).head(50)

**Findings:** `id`, `member_id`, and `url` are 100% empty. The next tier of missingness — `hardship_*`,
`settlement_*`, `sec_app_*`, and the joint-applicant columns (`*_joint`) — reflects that those fields only
populate for the small subset of borrowers who entered hardship plans, debt settlement, or applied jointly.
These are >90% missing but not dropped here; several of them describe post-origination events and belong in
a future leakage review, not something to silently impute away at the audit stage.

In [ ]:
audit_path = settings.outputs_dir / "metrics" / "initial_data_audit.csv"
audit_path.parent.mkdir(parents=True, exist_ok=True)
audit_df.to_csv(audit_path, index=False)

## Target variable

LendingClub's `loan_status` has nine values, most of which describe loans that haven't resolved yet.

In [ ]:
df["loan_status"].value_counts(dropna=False)

Only `Fully Paid` and `Charged Off` represent a resolved outcome — `Current`, `Late (*)`, `In Grace Period`,
and `Default` are still open loans and carry no ground truth yet. `clean_and_label` filters to the resolved
subset and maps `Charged Off → 1`, `Fully Paid → 0` as `target_default`.

In [ ]:
labeled_df = clean_and_label(df)
print("Labeled shape:", labeled_df.shape)
labeled_df["target_default"].value_counts(normalize=True) * 100

## Target distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(x="target_default", data=labeled_df)
plt.title("Target Class Distribution")
plt.xlabel("Default Flag")
plt.ylabel("Count")
plt.show()

**Summary:** ~1.30M loans have a resolved outcome, with a ~20% default rate — a real but moderate class
imbalance, handled downstream via `class_weight='balanced'` / `scale_pos_weight` rather than resampling.
Synthetic oversampling (SMOTE) is deliberately avoided here, since it tends to fabricate unrealistic
borrowers on tabular credit data.